# Feature Engineering — Customer Churn

**Purpose:** Transform raw cleaned data into model-ready features.  
We read the two separate CSVs from `data/02_intermediate/`, engineer  
features from each, aggregate BoB to the customer level, join them  
with the retention summary, and save one final CSV.

**Output:** `data/03_primary/churn_features.csv`

**Churn targets created:**
- `agreement_churn` — 1 if `Resolution Status == 'Customer Lost'`
- `customer_churn` — 1 if ALL of a customer's cases are 'Customer Lost'

In [1]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)

In [2]:
# ── Load cleaned data ────────────────────────────────────
retention = pd.read_csv('../../data/02_intermediate/retention_data.csv')
bob       = pd.read_csv('../../data/02_intermediate/bob_data.csv')

print(f"Retention : {retention.shape}")
print(f"BoB       : {bob.shape}")

Retention : (45806, 28)
BoB       : (65883, 26)


---
## Step 1 — Define Churn Targets

In [3]:
# ── 1a. Agreement-level churn flag ───────────────────────
# WHY: 'Customer Lost' in Resolution Status means the
# agreement associated with this case was lost.

retention['agreement_churn'] = (
    retention['Resolution Status'] == 'Customer Lost'
).astype(int)

print("Agreement churn distribution:")
print(retention['agreement_churn'].value_counts())

Agreement churn distribution:
agreement_churn
0    31320
1    14486
Name: count, dtype: int64


In [4]:
# ── 1b. Customer-level churn flag ────────────────────────
# WHY: A customer is fully churned only when ALL their
# retention cases resulted in 'Customer Lost'.  This is
# the stricter, business-critical definition.

cust_churn = retention.groupby('Customer Account Number').agg(
    total_cases   = ('agreement_churn', 'count'),
    churned_cases = ('agreement_churn', 'sum')
).reset_index()

cust_churn['customer_churn'] = (
    cust_churn['churned_cases'] == cust_churn['total_cases']
).astype(int)

print("Customer churn distribution:")
print(cust_churn['customer_churn'].value_counts())

Customer churn distribution:
customer_churn
0    16644
1     4882
Name: count, dtype: int64


---
## Step 2 — Date Features (Retention)

Date columns carry temporal patterns — we convert them into  
numeric durations and calendar components.

In [5]:
# ── 2a. Parse date columns ───────────────────────────────
# WHY: They arrive as strings; we need datetime objects
# to calculate durations.

# Case Creation Date has format "dd-mm-yyyy HH:MM"
retention['case_creation_dt'] = pd.to_datetime(
    retention['Case Creation Date'], dayfirst=True, errors='coerce'
)

# Registered & Resolved dates have format "dd-mm-yyyy"
retention['registered_dt'] = pd.to_datetime(
    retention['Registered Date'], dayfirst=True, errors='coerce'
)
retention['resolved_dt'] = pd.to_datetime(
    retention['Resolved Date'], dayfirst=True, errors='coerce'
)

# Expected Pull Date has format "dd-Mon-yy"
retention['expected_pull_dt'] = pd.to_datetime(
    retention['Expected Pull Date'], dayfirst=True, errors='coerce'
)

print("Date columns parsed.")
retention[['case_creation_dt', 'registered_dt',
           'resolved_dt', 'expected_pull_dt']].head()

Date columns parsed.


,case_creation_dt,registered_dt,resolved_dt,expected_pull_dt
0,2025-10-21 15:02:00,2025-10-21,NaT,2026-03-29
1,2025-10-21 15:00:00,2025-10-21,NaT,2026-03-29
2,2025-10-21 15:01:00,2025-10-21,NaT,2026-03-29
3,2025-08-15 13:44:00,2025-08-15,NaT,2026-07-12
4,2026-02-18 14:40:00,2026-02-18,NaT,2026-10-31


In [6]:
# ── 2b. Removed duration features ────────────────────────
# TARGET LEAKAGE FIX: 
# 'case_resolution_days' and 'days_to_expected_pull' are 
# calculated using dates that happen AT or AFTER the churn event.
# Including these would cause data leakage.
print("Target-leaking duration features removed.")


Target-leaking duration features removed.


In [7]:
# ── 2c. Calendar features ────────────────────────────────
# WHY: Churn behaviour may follow seasonal patterns
# (e.g. more cancellations at end of financial year).

retention['case_creation_month'] = retention['case_creation_dt'].dt.month
retention['case_creation_dayofweek'] = retention['case_creation_dt'].dt.dayofweek

print("Calendar features created.")

Calendar features created.


---
## Step 3 — VAN Features (Retention)

VAN = Value Added Number (contract value). Changes in VAN can  
signal customer dissatisfaction.

In [8]:
# ── 3a. VAN change flag ──────────────────────────────────
# WHY: If the Pull VAN differs from New VAN, the contract
# value has changed — possibly a downgrade signalling churn risk.

retention['van_changed'] = (
    retention['Pull VAN'] != retention['New VAN']
).astype(int)

print("VAN changed distribution:")
print(retention['van_changed'].value_counts())

VAN changed distribution:
van_changed
1    31935
0    13871
Name: count, dtype: int64


---
## Step 4 — Service Stress Features (Retention)

Overdue services and frequent repairs are signs of poor customer  
experience, which drives churn.

In [9]:
# ── 4a. Has overdue services ─────────────────────────────
# WHY: Even one overdue service visit can frustrate a customer
# and push them towards cancellation.

retention['has_overdue_services'] = (
    retention['Number of OverdueServices'].fillna(0) > 0
).astype(int)

print("Has overdue services distribution:")
print(retention['has_overdue_services'].value_counts())

Has overdue services distribution:
has_overdue_services
0    45806
Name: count, dtype: int64


In [10]:
# ── 4b. Repair to machine ratio ──────────────────────────
# WHY: A high ratio means many repairs per machine, indicating
# reliability issues — a strong churn driver.

retention['repair_to_machine_ratio'] = np.where(
    retention['Machines'].fillna(0) > 0,
    retention['Number Of Repair Cases'].fillna(0) / retention['Machines'],
    0  # avoid division by zero
)

print("Repair-to-machine ratio stats:")
print(retention['repair_to_machine_ratio'].describe())

Repair-to-machine ratio stats:
count    45806.000000
mean         0.031664
std          0.170059
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          4.000000
Name: repair_to_machine_ratio, dtype: float64


---
## Step 5 — Date Features (BoB)

In [11]:
# ── 5a. Agreement duration in days ───────────────────────
# WHY: Longer agreements might indicate greater customer
# commitment and lower churn risk.

bob['agreement_start_dt'] = pd.to_datetime(
    bob['agreement_start_date'], errors='coerce'
)
bob['agreement_end_dt'] = pd.to_datetime(
    bob['agreement_end_date'], errors='coerce'
)

bob['agreement_duration_days'] = (
    bob['agreement_end_dt'] - bob['agreement_start_dt']
).dt.days

print("Agreement duration stats (days):")
print(bob['agreement_duration_days'].describe())

Agreement duration stats (days):
count    65883.000000
mean      1778.976489
std        964.943219
min         61.000000
25%       1095.000000
50%       1460.000000
75%       2556.000000
max       4749.000000
Name: agreement_duration_days, dtype: float64


---
## Step 6 — BoB Customer-Level Aggregations

The BoB data is at the agreement level. We aggregate it to  
one row per customer so we can join it to the retention summary.

In [12]:
# ── 6a. Aggregate BoB features per customer ──────────────
# WHY: The model needs one row per customer.  We summarise
# the customer's portfolio from the BoB data.

bob_agg = bob.groupby('account_number').agg(

    # Total number of unique agreements the customer has
    total_agreements     = ('agreement_number', 'nunique'),

    # Total annual contract value — higher value = more at stake
    total_annual_value   = ('unit_amount', 'sum'),

    # Average agreement value — gives a sense of deal size
    avg_agreement_value  = ('unit_amount', 'mean'),

    # Average agreement duration — longer = more committed
    avg_agreement_duration = ('agreement_duration_days', 'mean'),

    # How many different product lines the customer uses
    num_lines_of_business = ('line_of_business', 'nunique'),

    # How many different machine types — more diversity = stickier
    num_machine_types    = ('machine', 'nunique'),

    # Total BoB value (product + fee)
    total_bob_value      = ('total_bob', 'sum'),

    # Average product BoB per agreement
    avg_product_bob      = ('product_bob', 'mean'),

).reset_index()

# Rename the key column to match the retention data
bob_agg.rename(columns={'account_number': 'Customer Account Number'},
               inplace=True)

print(f"BoB aggregated to {bob_agg.shape[0]} unique customers.")
bob_agg.head()

BoB aggregated to 23772 unique customers.


,Customer Account Number,total_agreements,total_annual_value,avg_agreement_value,avg_agreement_duration,num_lines_of_business,num_machine_types,total_bob_value,avg_product_bob
0,GBA221737,1,0.00,NaN,365.0,1,0,8680.10,8680.100
1,UK02-000167723,2,0.00,NaN,1096.0,1,0,3616.75,1808.375
2,UK02-000167746,1,236.67,236.67,730.0,1,1,2999.99,2840.030
3,UK02-000167747,1,0.00,NaN,365.0,1,0,2540.14,2540.140
4,UK02-000167753,1,361.19,361.19,1096.5,1,1,8948.50,4334.270


---
## Step 7 — Aggregate Retention to Customer Level

In [13]:
# ── 7a. Retention summary per customer ───────────────────
# WHY: We need one row per customer.  We keep the most
# useful numeric and categorical info from the retention cases.

ret_agg = retention.groupby('Customer Account Number').agg(

    # Churn target
    total_ret_cases        = ('agreement_churn', 'count'),
    churned_cases          = ('agreement_churn', 'sum'),

    # Average contract value (VAN) across all cases
    avg_van                = ('VAN', 'mean'),
    total_van              = ('VAN', 'sum'),

    # Contract & machine counts (take max since they're customer-level)
    num_contracts          = ('Number of Contracts', 'max'),
    num_machines           = ('Machines', 'max'),

    # Service stress indicators
    total_repair_cases     = ('Number Of Repair Cases', 'sum'),
    total_overdue_services = ('Number of OverdueServices', 'sum'),
    has_any_overdue        = ('has_overdue_services', 'max'),
    avg_repair_ratio       = ('repair_to_machine_ratio', 'mean'),

    # VAN change
    any_van_changed        = ('van_changed', 'max'),

    # Duration features removed to prevent target leakage

    # Calendar features (mode approximated by median)
    typical_creation_month = ('case_creation_month', 'median'),

).reset_index()

print(f"Retention aggregated to {ret_agg.shape[0]} unique customers.")
ret_agg.head()

Retention aggregated to 21526 unique customers.


,Customer Account Number,total_ret_cases,churned_cases,avg_van,total_van,num_contracts,num_machines,total_repair_cases,total_overdue_services,has_any_overdue,avg_repair_ratio,any_van_changed,typical_creation_month
0,IE01-C2042480-L,1,1,1242.908615,1242.908615,1.0,1.0,0.0,0.0,0,0.0,0,7.0
1,IE01-C2048834-L,1,0,1676.516270,1676.516270,1.0,1.0,0.0,0.0,0,0.0,0,11.0
2,IE01-C2156428-L,2,0,2632.461364,5264.922728,1.0,1.0,0.0,0.0,0,0.0,0,9.5
3,IE01-C2244318-L,1,0,3178.122870,3178.122870,1.0,1.0,0.0,0.0,0,0.0,0,10.0
4,IE01-C2270501-L,5,0,843.777914,4218.889571,1.0,1.0,0.0,0.0,0,0.0,0,10.0


In [14]:
# ── 7b. Add customer-level churn flag ────────────────────
# WHY: customer_churn = 1 only when ALL cases for the
# customer ended in 'Customer Lost'.

ret_agg['agreement_churn'] = (ret_agg['churned_cases'] > 0).astype(int)
ret_agg['customer_churn']  = (
    ret_agg['churned_cases'] == ret_agg['total_ret_cases']
).astype(int)

print("Customer churn:")
print(ret_agg['customer_churn'].value_counts())

Customer churn:
customer_churn
0    16644
1     4882
Name: count, dtype: int64


In [15]:
# ── 7c. Keep the most common categorical values per customer ─
# WHY: Categorical features like CompanySize and Customer Tier
# are useful for the model.  For customer-level data we take
# the most frequent (mode) value across that customer's cases.

cat_cols_to_keep = ['CompanySize', 'Customer Tier', 'Case Origin']
cat_cols_to_keep = [c for c in cat_cols_to_keep if c in retention.columns]

for col in cat_cols_to_keep:
    # Get the mode (most common value) per customer
    mode_df = (
        retention.groupby('Customer Account Number')[col]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
        .reset_index()
    )
    ret_agg = ret_agg.merge(mode_df, on='Customer Account Number', how='left')

print(f"Categorical columns added: {cat_cols_to_keep}")
ret_agg.head()

Categorical columns added: ['CompanySize', 'Customer Tier', 'Case Origin']


,Customer Account Number,total_ret_cases,churned_cases,avg_van,total_van,num_contracts,num_machines,total_repair_cases,total_overdue_services,has_any_overdue,avg_repair_ratio,any_van_changed,typical_creation_month,agreement_churn,customer_churn,CompanySize,Customer Tier,Case Origin
0,IE01-C2042480-L,1,1,1242.908615,1242.908615,1.0,1.0,0.0,0.0,0,0.0,0,7.0,1,1,>1000,NaN,Customer Email
1,IE01-C2048834-L,1,0,1676.516270,1676.516270,1.0,1.0,0.0,0.0,0,0.0,0,11.0,0,0,500-999,NaN,Proactive Prevention
2,IE01-C2156428-L,2,0,2632.461364,5264.922728,1.0,1.0,0.0,0.0,0,0.0,0,9.5,0,0,>1000,NaN,Customer Email
3,IE01-C2244318-L,1,0,3178.122870,3178.122870,1.0,1.0,0.0,0.0,0,0.0,0,10.0,0,0,Oct-19,NaN,Internal Email
4,IE01-C2270501-L,5,0,843.777914,4218.889571,1.0,1.0,0.0,0.0,0,0.0,0,10.0,0,0,Oct-19,NaN,Customer Call


---
## Step 8 — Join BoB Aggregates to Retention Summary

In [16]:
# ── 8a. Merge on Customer Account Number ─────────────────
# WHY: This gives us ONE table with both retention behaviour
# and portfolio (BoB) characteristics per customer.
# We use left join so we keep all retention customers even if
# they don't appear in BoB (they'll get NaN for BoB columns).

features = ret_agg.merge(bob_agg, on='Customer Account Number', how='left')

print(f"Final features shape: {features.shape}")
features.head()

Final features shape: (21526, 26)


,Customer Account Number,total_ret_cases,churned_cases,avg_van,total_van,num_contracts,num_machines,total_repair_cases,total_overdue_services,has_any_overdue,avg_repair_ratio,any_van_changed,typical_creation_month,agreement_churn,customer_churn,CompanySize,Customer Tier,Case Origin,total_agreements,total_annual_value,avg_agreement_value,avg_agreement_duration,num_lines_of_business,num_machine_types,total_bob_value,avg_product_bob
0,IE01-C2042480-L,1,1,1242.908615,1242.908615,1.0,1.0,0.0,0.0,0,0.0,0,7.0,1,1,>1000,NaN,Customer Email,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IE01-C2048834-L,1,0,1676.516270,1676.516270,1.0,1.0,0.0,0.0,0,0.0,0,11.0,0,0,500-999,NaN,Proactive Prevention,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IE01-C2156428-L,2,0,2632.461364,5264.922728,1.0,1.0,0.0,0.0,0,0.0,0,9.5,0,0,>1000,NaN,Customer Email,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IE01-C2244318-L,1,0,3178.122870,3178.122870,1.0,1.0,0.0,0.0,0,0.0,0,10.0,0,0,Oct-19,NaN,Internal Email,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IE01-C2270501-L,5,0,843.777914,4218.889571,1.0,1.0,0.0,0.0,0,0.0,0,10.0,0,0,Oct-19,NaN,Customer Call,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## Step 9 — Encode Categorical Features

In [17]:
# ── 9a. Ordinal encode CompanySize ───────────────────────
# WHY: Company size has a natural order (small → large).
# Ordinal encoding preserves this ordering for the model.

size_order = {
    '1-9': 1, '10-19': 2, '20-49': 3, '50-99': 4,
    '100-249': 5, '250-499': 6, '500-999': 7,
    '>=1000': 8, '>1000': 8
}

if 'CompanySize' in features.columns:
    features['company_size_encoded'] = (
        features['CompanySize'].map(size_order)
    )
    print("CompanySize encoded.")
    print(features['company_size_encoded'].value_counts().sort_index())

CompanySize encoded.
company_size_encoded
3.0    1400
4.0    1278
5.0    1324
6.0     764
7.0     486
8.0    3567
Name: count, dtype: int64


In [18]:
# ── 9b. One-hot encode Customer Tier and Case Origin ─────
# WHY: These are nominal categories (no natural order).
# One-hot encoding creates binary columns for each category.
# drop_first=True avoids the multicollinearity trap.

ohe_cols = ['Customer Tier', 'Case Origin']
ohe_cols = [c for c in ohe_cols if c in features.columns]

features = pd.get_dummies(features, columns=ohe_cols, drop_first=True)

print(f"Shape after one-hot encoding: {features.shape}")

Shape after one-hot encoding: (21526, 59)


---
## Step 10 — Final Cleanup & Save

In [19]:
# ── 10a. Drop ID and raw text columns ────────────────────
# WHY: The model should learn patterns, not memorise IDs.
# We also drop the original CompanySize since we encoded it.

drop_cols = ['Customer Account Number', 'CompanySize']
drop_cols = [c for c in drop_cols if c in features.columns]

features.drop(columns=drop_cols, inplace=True)

print(f"Final shape: {features.shape}")
print(f"\nColumns:")
print(features.columns.tolist())

Final shape: (21526, 57)

Columns:
['total_ret_cases', 'churned_cases', 'avg_van', 'total_van', 'num_contracts', 'num_machines', 'total_repair_cases', 'total_overdue_services', 'has_any_overdue', 'avg_repair_ratio', 'any_van_changed', 'typical_creation_month', 'agreement_churn', 'customer_churn', 'total_agreements', 'total_annual_value', 'avg_agreement_value', 'avg_agreement_duration', 'num_lines_of_business', 'num_machine_types', 'total_bob_value', 'avg_product_bob', 'company_size_encoded', 'Customer Tier_Key Account', 'Customer Tier_Platinum', 'Customer Tier_Platinum +', 'Customer Tier_Platinum+', 'Case Origin_BSU/ERP customer services', 'Case Origin_Branch Administrator', 'Case Origin_Branch Manager', 'Case Origin_Collectable Services Manager', 'Case Origin_Complaint Case', 'Case Origin_Credit Controller', 'Case Origin_Customer Call', 'Case Origin_Customer Email', 'Case Origin_Customer Letter', 'Case Origin_Customer Negotiation', 'Case Origin_Customer Service Manager', 'Case Origin_

In [20]:
# ── 10b. Quick look at the final dataset ─────────────────
features.info()
print("\n")
features.describe()

<class 'pandas.DataFrame'>
RangeIndex: 21526 entries, 0 to 21525
Data columns (total 57 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   total_ret_cases                           21526 non-null  int64  
 1   churned_cases                             21526 non-null  int64  
 2   avg_van                                   21502 non-null  float64
 3   total_van                                 21526 non-null  float64
 4   num_contracts                             13980 non-null  float64
 5   num_machines                              13980 non-null  float64
 6   total_repair_cases                        21526 non-null  float64
 7   total_overdue_services                    21526 non-null  float64
 8   has_any_overdue                           21526 non-null  int64  
 9   avg_repair_ratio                          21526 non-null  float64
 10  any_van_changed                           215

,total_ret_cases,churned_cases,avg_van,total_van,num_contracts,num_machines,total_repair_cases,total_overdue_services,has_any_overdue,avg_repair_ratio,any_van_changed,typical_creation_month,agreement_churn,customer_churn,total_agreements,total_annual_value,avg_agreement_value,avg_agreement_duration,num_lines_of_business,num_machine_types,total_bob_value,avg_product_bob,company_size_encoded
count,21526.000000,21526.000000,21502.000000,2.152600e+04,13980.000000,13980.000000,21526.000000,21526.0,21526.0,21526.00000,21526.000000,21526.000000,21526.000000,21526.000000,5628.000000,5628.000000,5609.000000,5628.000000,5628.000000,5628.000000,5628.000000,5628.000000,8819.000000
mean,2.127938,0.672954,2788.614422,7.913183e+03,1.172747,2.311660,0.346929,0.0,0.0,0.02547,0.812274,6.269349,0.570752,0.226796,2.727079,652.011702,227.559604,1626.344457,1.464286,1.483475,7457.369570,2431.786054,5.947840
std,3.219624,0.771068,6030.826540,4.548794e+04,0.718762,3.589824,5.198218,0.0,0.0,0.15097,0.390503,3.183825,0.494980,0.418769,8.345192,2108.926653,329.614665,881.875448,0.683896,1.418070,24541.639933,3078.882866,1.956838
min,1.000000,0.000000,0.000000,0.000000e+00,1.000000,1.000000,0.000000,0.0,0.0,0.00000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,5.250000,267.000000,1.000000,0.000000,0.000000,0.000000,3.000000
25%,1.000000,0.000000,412.667083,6.677580e+02,1.000000,1.000000,0.000000,0.0,0.0,0.00000,1.000000,4.000000,0.000000,0.000000,1.000000,126.057500,72.410000,1094.000000,1.000000,1.000000,1439.835000,818.528000,4.000000
50%,2.000000,1.000000,1246.850178,2.033738e+03,1.000000,1.000000,0.000000,0.0,0.0,0.00000,1.000000,6.000000,1.000000,0.000000,2.000000,245.265000,144.040000,1277.500000,1.000000,1.000000,2732.120000,1604.140000,6.000000
75%,2.000000,1.000000,2892.855740,5.385344e+03,1.000000,2.000000,0.000000,0.0,0.0,0.00000,1.000000,9.000000,1.000000,0.000000,3.000000,523.835000,276.575000,2191.000000,2.000000,2.000000,5954.425000,3003.200000,8.000000
max,377.000000,29.000000,177731.741500,4.456335e+06,35.000000,105.000000,528.000000,0.0,0.0,4.00000,1.000000,12.000000,1.000000,1.000000,518.000000,68124.370000,7492.520000,4382.000000,5.000000,29.000000,818018.390000,76431.680000,8.000000


In [21]:
# ── 10c. Save to data/03_primary ─────────────────────────
output_path = '../../data/03_primary/churn_features.csv'
features.to_csv(output_path, index=False)

print(f"Saved {features.shape[0]} rows × {features.shape[1]} columns")
print(f"to {output_path}")

Saved 21526 rows × 57 columns
to ../../data/03_primary/churn_features.csv
